# Exported Model Usage + Real-Time Performance

This notebook uses the exported best models through `model_realtime_eval.py` and prints live metrics every time you run the evaluation cells.

In [1]:
import json
import subprocess
from pathlib import Path

ROOT = Path.cwd()
VENV_PY = ROOT / '.venv' / 'Scripts' / 'python.exe'
EVAL_SCRIPT = ROOT / 'model_realtime_eval.py'

if not VENV_PY.exists():
    raise FileNotFoundError(f'Python env not found: {VENV_PY}')
if not EVAL_SCRIPT.exists():
    raise FileNotFoundError(f'Eval script not found: {EVAL_SCRIPT}')

print('Runtime OK')
print('Python:', VENV_PY)
print('Script:', EVAL_SCRIPT)

Runtime OK
Python: e:\football-pred\.venv\Scripts\python.exe
Script: e:\football-pred\model_realtime_eval.py


In [2]:
def run_eval(mode, dataset='combined', test_size=0.2, seed=42, n=10):
    cmd = [
        str(VENV_PY),
        str(EVAL_SCRIPT),
        '--mode', mode,
        '--dataset', dataset,
        '--test-size', str(test_size),
        '--seed', str(seed),
        '--n', str(n),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return json.loads(proc.stdout.strip())

def print_matrix(labels, matrix):
    print('labels:', labels)
    print('confusion_matrix:')
    for row in matrix:
        print(row)

## Cell 4: Example Test Data Predictions

Runs sample predictions using both exported classification models.

In [3]:
combined_sample = run_eval(mode='sample', dataset='combined', n=10, seed=42)
detailed_sample = run_eval(mode='sample', dataset='detailed', n=10, seed=42)

print('Combined sample predictions:')
for row in combined_sample['rows']:
    print(row)

print('\nDetailed sample predictions:')
for row in detailed_sample['rows']:
    print(row)

Combined sample predictions:
{'index': 4751, 'true_pos': 'DF', 'pred_pos': 'DF'}
{'index': 3541, 'true_pos': 'DF', 'pred_pos': 'DF'}
{'index': 907, 'true_pos': 'FW', 'pred_pos': 'FW'}
{'index': 2833, 'true_pos': 'DF', 'pred_pos': 'DF'}
{'index': 3106, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 4635, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 2244, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 1924, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 3804, 'true_pos': 'GK', 'pred_pos': 'GK'}
{'index': 2634, 'true_pos': 'GK', 'pred_pos': 'GK'}

Detailed sample predictions:
{'index': 2015, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 308, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 1815, 'true_pos': 'GK', 'pred_pos': 'GK'}
{'index': 521, 'true_pos': 'FW', 'pred_pos': 'FW'}
{'index': 74, 'true_pos': 'MF', 'pred_pos': 'MF'}
{'index': 1752, 'true_pos': 'DF', 'pred_pos': 'DF'}
{'index': 1284, 'true_pos': 'DF', 'pred_pos': 'DF'}
{'index': 2299, 'true_pos': 'FW', 'pred_pos': 'FW'}
{'index': 

## Cell 6: Real-Time Classification Accuracy

Change `dataset`, `test_size`, or `seed`, then rerun this cell to get updated real-time accuracy.

In [6]:
dataset = 'combined'   # 'combined' or 'detailed'
test_size = 0.2
seed = 42

cls = run_eval(mode='classification', dataset=dataset, test_size=test_size, seed=seed)
print(f"Model: {cls['model']}")
print(f"Train accuracy: {cls['train_accuracy']:.4f}")
print(f"Test rows: {cls['n_test']}")
print(f"Accuracy (real-time): {cls['accuracy']:.4f}")
print(f"Cross-validation accuracy: {cls['cv_accuracy_mean']:.4f} +/- {cls['cv_accuracy_std']:.4f}")
print(f"Macro-F1: {cls['macro_f1']:.4f}")
print('Per-class accuracy:')
for label, value in cls['per_class_accuracy'].items():
    print(f"  {label}: {value:.4f}")
print_matrix(cls['labels'], cls['confusion_matrix'])

Model: combined_base_HistGradientBoosting
Train accuracy: 0.9939
Test rows: 981
Accuracy (real-time): 0.8002
Cross-validation accuracy: 0.8036 +/- 0.0091
Macro-F1: 0.8363
Per-class accuracy:
  DF: 0.8110
  FW: 0.8678
  GK: 0.9863
  MF: 0.6877
labels: ['DF', 'FW', 'GK', 'MF']
confusion_matrix:
[296, 11, 0, 58]
[16, 210, 0, 16]
[1, 0, 72, 0]
[48, 46, 0, 207]


## Cell 8: Real-Time Regression Performance

Rerun this cell for updated MSE/R2 when you change split settings.

In [8]:
test_size = 0.2
seed = 42

reg = run_eval(mode='regression', test_size=test_size, seed=seed)
print(f"Model: {reg['model']}")
print(f"Train R2: {reg['train_r2']:.4f}")
print(f"Test rows: {reg['n_test']}")
print(f"Holdout R2: {reg['holdout_r2']:.4f}")
print(f"MSE (real-time): {reg['mse']:.4f}")
print(f"R2 (real-time): {reg['r2']:.4f}")
print(f"Cross-validation R2: {reg['cv_r2_mean']:.4f} +/- {reg['cv_r2_std']:.4f}")
print('Preview:')
for row in reg['preview']:
    print(row)

Model: combined_base_regression_ElasticNet
Train R2: 0.8803
Test rows: 981
Holdout R2: 0.8761
MSE (real-time): 1.3664
R2 (real-time): 0.8761
Cross-validation R2: 0.8687 +/- 0.0132
Preview:
{'actual_gls': 1.0, 'pred_gls': 2.017}
{'actual_gls': 0.0, 'pred_gls': 0.539}
{'actual_gls': 3.0, 'pred_gls': 3.743}
{'actual_gls': 0.0, 'pred_gls': 0.416}
{'actual_gls': 4.0, 'pred_gls': 4.809}
{'actual_gls': 0.0, 'pred_gls': 0.186}
{'actual_gls': 0.0, 'pred_gls': -0.243}
{'actual_gls': 0.0, 'pred_gls': 0.037}
{'actual_gls': 0.0, 'pred_gls': -0.281}
{'actual_gls': 0.0, 'pred_gls': 0.162}
